In [1]:
# Cell 1: imports, paths, and settings

import shutil
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
TASK_ROOT = Path.cwd().parent
INPUT = TASK_ROOT / 'input'
OUTPUT = TASK_ROOT / 'output'
FIT = INPUT / 'fit'
OUTPUT.mkdir(parents=True, exist_ok=True)
EPS = 0.49
N = 50000
Y = 20
MASTER_SEED = 63
SAMPLE_N = 500
WRITE_WIDE_TRAJECTORY_CSV = False
MODEL_ORDER = ['arw4', 'unfitted_grw', 'grw_y', 'grw_global', 'ar1_y', 'ar1_s', 'ar1_s_globalinit', 'hurdle_ar1_s', 'hurdle_ar1_s_p', 'hurdle_ar3_s_p']
MODEL_NAMES = {'arw4': 'ARW4', 'unfitted_grw': 'Unfitted-GRW', 'grw_y': 'GRW-Y', 'grw_global': 'GRW-G', 'ar1_y': 'AR(1)-GRW-Y', 'ar1_s': 'AR(1)-GRW-S', 'ar1_s_globalinit': 'AR(1)-GRW-S-G', 'hurdle_ar1_s': 'Hurdle-AR(1)-GRW-S', 'hurdle_ar1_s_p': 'Hurdle-AR(1)-GRW-S-P', 'hurdle_ar3_s_p': 'Hurdle-AR(3)-GRW-S-P'}
seed_sequence = np.random.SeedSequence(MASTER_SEED)
child_sequences = seed_sequence.spawn(len(MODEL_ORDER))
MODEL_SEEDS = {model: int(seed.generate_state(1, dtype=np.uint64)[0]) for model, seed in zip(MODEL_ORDER, child_sequences)}
missing_model_dirs = [model for model in MODEL_ORDER if not (FIT / model).is_dir()]
if missing_model_dirs:
    raise FileNotFoundError(f'Missing fit directories: {missing_model_dirs}')
print(f'Fit input: {FIT.resolve()}')
print(f'Simulation output: {OUTPUT.resolve()}')
print(f'N={N:,}; years=0-{Y}; master seed={MASTER_SEED}')


Fit input: /Users/samlunemagid/Desktop/SPAARW2/fit/output
Simulation output: /Users/samlunemagid/Desktop/SPAARW2/simulate/output
N=50,000; years=0-20; master seed=63


In [2]:
# Cell 2: shared load, stage, and draw helpers

def read_table(model_tag, filename):
    path = FIT / model_tag / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def read_one(model_tag, filename):
    table = read_table(model_tag, filename)
    if len(table) != 1:
        raise ValueError(f'Expected one row in {model_tag}/{filename}; found {len(table)}')
    return table.iloc[0]

def row_value(row, *columns):
    for column in columns:
        if column in row.index and np.isfinite(float(row[column])):
            return float(row[column])
    raise KeyError(f'None of {columns} found as finite values')

def stage_rows_by_year(table, y=Y):
    rows = [None] * y
    for _, row in table.iterrows():
        start = int(row['transition_year_start'])
        end = int(row['transition_year_end'])
        for year in range(start, end + 1):
            if not 0 <= year < y:
                continue
            if rows[year] is not None:
                raise ValueError(f'Overlapping parameter stages at transition year {year}')
            rows[year] = row
    missing = [year for year, row in enumerate(rows) if row is None]
    if missing:
        raise ValueError(f'Parameter table does not cover transition years: {missing}')
    return rows

def positive_exponential(scale, size, rng):
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError(f'Invalid exponential scale: {scale}')
    draws = rng.exponential(scale=scale, size=size)
    zero = draws <= 0
    while zero.any():
        draws[zero] = rng.exponential(scale=scale, size=int(zero.sum()))
        zero = draws <= 0
    return draws

def safe_exp(z):
    limits = np.log([np.finfo(float).tiny, np.finfo(float).max])
    return np.exp(np.clip(z, limits[0], limits[1]))

def q_from_offset_log(z):
    return np.maximum(safe_exp(z) - EPS, 0.0)

def positive_from_log(z):
    return safe_exp(z)

def logistic(x):
    x = np.clip(np.asarray(x, dtype=float), -40, 40)
    return 1 / (1 + np.exp(-x))

def sample_trunc_laplace_nonnegative(loc, scale, rng):
    loc = np.asarray(loc, dtype=float)
    if loc.size == 0:
        return np.array([], dtype=float)
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError(f'Invalid Laplace scale: {scale}')
    lower_cdf = stats.laplace.cdf(0, loc=loc, scale=scale)
    lower_cdf = np.clip(lower_cdf, 0, np.nextafter(1.0, 0.0))
    u = lower_cdf + (1 - lower_cdf) * rng.random(loc.size)
    u = np.minimum(u, np.nextafter(1.0, 0.0))
    return stats.laplace.ppf(u, loc=loc, scale=scale)

def resolve_params(stage_lookup, global_row, stage, columns):
    row = stage_lookup[stage]
    if all((np.isfinite(float(row[column])) for column in columns)):
        return row
    if all((np.isfinite(float(global_row[column])) for column in columns)):
        return global_row
    raise ValueError(f'No finite {columns} parameters for stage {stage}')


In [3]:
# Cell 3: plotting summaries and output helpers

def year_summary(trajs):
    rows = []
    for year in range(trajs.shape[0]):
        q = pd.Series(trajs[year])
        log_q = np.log(q + EPS)
        q_pos = q[q > 0]
        log_q_pos = np.log(q_pos)
        rows.append({'year': year, 'n': len(q), 'zero_fraction': q.eq(0).mean(), 'mean_productivity': q.mean(), 'median_productivity': q.median(), 'var_productivity': q.var(ddof=0), 'mean_log_productivity': log_q.mean(), 'var_log_productivity': log_q.var(ddof=0), 'mean_positive_log_productivity': log_q_pos.mean(), 'var_positive_log_productivity': log_q_pos.var(ddof=0), 'q25_productivity': q.quantile(0.25), 'q75_productivity': q.quantile(0.75), 'q90_productivity': q.quantile(0.9), 'q95_productivity': q.quantile(0.95)})
    return pd.DataFrame(rows)

def transition_summary(trajs):
    raw_delta = trajs[1:] - trajs[:-1]
    log_delta = np.log(trajs[1:] + EPS) - np.log(trajs[:-1] + EPS)
    rows = []
    for year in range(trajs.shape[0] - 1):
        rows.append({'transition_year': year, 'n': trajs.shape[1], 'mean_raw_delta': raw_delta[year].mean(), 'median_raw_delta': np.median(raw_delta[year]), 'var_raw_delta': raw_delta[year].var(ddof=0), 'mean_log_delta': log_delta[year].mean(), 'median_log_delta': np.median(log_delta[year]), 'var_log_delta': log_delta[year].var(ddof=0)})
    return pd.DataFrame(rows)

def trajectory_sample(trajs, sample_n=SAMPLE_N):
    sample_n = min(sample_n, trajs.shape[1])
    columns = [f'year_{year}' for year in range(trajs.shape[0])]
    sample = pd.DataFrame(trajs[:, :sample_n].T, columns=columns)
    sample.insert(0, 'ix', np.arange(sample_n))
    return sample

def trajectory_long_sample(trajs, sample_n=SAMPLE_N):
    sample_n = min(sample_n, trajs.shape[1])
    sampled = trajs[:, :sample_n]
    years = np.arange(trajs.shape[0])
    ix = np.arange(sample_n)
    frame = pd.DataFrame({'CareerAge': np.repeat(years, sample_n), 'ix': np.tile(ix, len(years)), 'pubs_adj': sampled.reshape(-1)})
    frame['pubs_adj_next'] = frame.groupby('ix')['pubs_adj'].shift(-1)
    frame['q_adj_delta'] = frame['pubs_adj_next'] - frame['pubs_adj']
    frame['log_pubs_adj'] = np.log(frame['pubs_adj'] + EPS)
    frame['log_pubs_next'] = np.log(frame['pubs_adj_next'] + EPS)
    frame['log_delta'] = frame['log_pubs_next'] - frame['log_pubs_adj']
    return frame

def copy_parameter_tables(model_tag, model_output):
    parameter_output = model_output / 'parameters'
    parameter_output.mkdir(parents=True, exist_ok=True)
    rows = []
    for source in sorted((FIT / model_tag).glob('*.csv')):
        destination = parameter_output / source.name
        shutil.copy2(source, destination)
        table = pd.read_csv(source)
        rows.append({'model_tag': model_tag, 'file': source.name, 'relative_path': str(destination.relative_to(OUTPUT)), 'rows': len(table), 'columns': ','.join(table.columns)})
    if not rows:
        raise FileNotFoundError(f'No parameter CSVs found for {model_tag}')
    return rows

def save_model_output(model_tag, trajs, seed, active=None, active_run_length=None):
    trajs = np.asarray(trajs, dtype=float)
    expected_shape = (Y + 1, N)
    if trajs.shape != expected_shape:
        raise AssertionError(f'{model_tag}: expected {expected_shape}, found {trajs.shape}')
    if not np.isfinite(trajs).all():
        raise AssertionError(f'{model_tag}: nonfinite trajectory values')
    if (trajs < 0).any():
        raise AssertionError(f'{model_tag}: negative trajectory values')
    model_output = OUTPUT / model_tag
    model_output.mkdir(parents=True, exist_ok=True)
    np.save(model_output / 'trajectories.npy', trajs, allow_pickle=False)
    year_summary(trajs).to_csv(model_output / 'year-summary.csv', index=False)
    transition_summary(trajs).to_csv(model_output / 'transition-summary.csv', index=False)
    trajectory_sample(trajs).to_csv(model_output / 'trajectory-sample.csv', index=False)
    trajectory_long_sample(trajs).to_csv(model_output / 'trajectory-long-sample.csv', index=False)
    wide_csv = ''
    if WRITE_WIDE_TRAJECTORY_CSV:
        wide_csv = 'trajectories.csv.gz'
        pd.DataFrame(trajs.T, columns=[f'year_{year}' for year in range(Y + 1)]).to_csv(model_output / wide_csv, index_label='ix', compression='gzip')
    active_file = ''
    if active is not None:
        active = np.asarray(active, dtype=bool)
        if active.shape != expected_shape:
            raise AssertionError(f'{model_tag}: invalid active-state shape')
        if not np.array_equal(active, trajs > 0):
            raise AssertionError(f'{model_tag}: active states and productivity disagree')
        active_file = 'active-states.npy'
        np.save(model_output / active_file, active, allow_pickle=False)
    run_length_file = ''
    if active_run_length is not None:
        active_run_length = np.asarray(active_run_length, dtype=np.int16)
        if active_run_length.shape != expected_shape:
            raise AssertionError(f'{model_tag}: invalid active-run-length shape')
        run_length_file = 'active-run-length.npy'
        np.save(model_output / run_length_file, active_run_length, allow_pickle=False)
    parameter_rows = copy_parameter_tables(model_tag, model_output)
    metadata = pd.DataFrame([{'model': MODEL_NAMES[model_tag], 'model_tag': model_tag, 'n_trajectories': N, 'career_year_start': 0, 'career_year_end': Y, 'master_seed': MASTER_SEED, 'model_seed': seed, 'eps': EPS, 'trajectory_orientation': 'rows=career years; columns=simulated scholars', 'trajectory_file': 'trajectories.npy', 'wide_trajectory_csv': wide_csv, 'active_state_file': active_file, 'active_run_length_file': run_length_file, 'year_summary_file': 'year-summary.csv', 'transition_summary_file': 'transition-summary.csv', 'trajectory_long_sample_file': 'trajectory-long-sample.csv'}])
    metadata.to_csv(model_output / 'simulation-metadata.csv', index=False)
    registry_row = {'model': MODEL_NAMES[model_tag], 'model_tag': model_tag, 'n_trajectories': N, 'n_years': Y + 1, 'model_seed': seed, 'trajectory_file': f'{model_tag}/trajectories.npy', 'active_state_file': f'{model_tag}/{active_file}' if active_file else '', 'active_run_length_file': f'{model_tag}/{run_length_file}' if run_length_file else '', 'year_summary_file': f'{model_tag}/year-summary.csv', 'transition_summary_file': f'{model_tag}/transition-summary.csv', 'trajectory_long_sample_file': f'{model_tag}/trajectory-long-sample.csv', 'minimum': trajs.min(), 'maximum': trajs.max(), 'zero_fraction_year_20': np.mean(trajs[Y] == 0)}
    return (registry_row, parameter_rows)
simulation_registry_rows = []
parameter_registry_rows = []


In [4]:
# Cell 4: simulate ARW4

model_tag = 'arw4'
rng = np.random.default_rng(MODEL_SEEDS[model_tag])
arw4_stage_params = read_table(model_tag, 'stage-parameters.csv')
arw4_global_params = read_table(model_tag, 'global-parameters.csv')
arw4_initialization_params = read_one(model_tag, 'initialization-parameters.csv')
arw4_by_year = stage_rows_by_year(arw4_stage_params)
trajs_arw4 = np.empty((Y + 1, N), dtype=float)
trajs_arw4[0] = rng.exponential(scale=float(arw4_initialization_params['scale']), size=N)
for t, params in enumerate(arw4_by_year):
    loc = float(params['mode_beta']) * trajs_arw4[t]
    trajs_arw4[t + 1] = sample_trunc_laplace_nonnegative(loc, float(params['alpha']), rng)
registry_row, parameter_rows = save_model_output(model_tag, trajs_arw4, MODEL_SEEDS[model_tag])
simulation_registry_rows.append(registry_row)
parameter_registry_rows.extend(parameter_rows)
print(MODEL_NAMES[model_tag], trajs_arw4.shape)


ARW4 (21, 50000)


In [5]:
# Cell 5: simulate unfitted GRW

model_tag = 'unfitted_grw'
rng = np.random.default_rng(MODEL_SEEDS[model_tag])
unfitted_grw_params = read_table(model_tag, 'fixed-parameters.csv')
fixed = unfitted_grw_params.set_index('parameter')['value'].to_dict()
x = np.full(N, float(fixed['x0']), dtype=float)
trajs_unfitted_grw = np.empty((Y + 1, N), dtype=float)
for t in range(Y + 1):
    x = x * np.exp(rng.normal(float(fixed['log_gamma_mean']), float(fixed['log_gamma_sigma']), size=N))
    trajs_unfitted_grw[t] = x
registry_row, parameter_rows = save_model_output(model_tag, trajs_unfitted_grw, MODEL_SEEDS[model_tag])
simulation_registry_rows.append(registry_row)
parameter_registry_rows.extend(parameter_rows)
print(MODEL_NAMES[model_tag], trajs_unfitted_grw.shape)


Unfitted-GRW (21, 50000)


In [6]:
# Cell 6: simulate GRW-Y

model_tag = 'grw_y'
rng = np.random.default_rng(MODEL_SEEDS[model_tag])
grw_y_params = read_table(model_tag, 'yearwise-parameters.csv').sort_values('transition_year')
grw_y_global_params = read_table(model_tag, 'global-parameters.csv')
grw_y_initialization_params = read_one(model_tag, 'initialization-parameters.csv')
if grw_y_params['transition_year'].tolist() != list(range(Y)):
    raise AssertionError('GRW-Y parameters must cover transition years 0-19')
trajs_grw_y = np.empty((Y + 1, N), dtype=float)
trajs_grw_y[0] = rng.exponential(scale=float(grw_y_initialization_params['scale']), size=N)
z = np.log(trajs_grw_y[0] + EPS)
for _, params in grw_y_params.iterrows():
    t = int(params['transition_year'])
    z_next = z + rng.normal(float(params['mu']), float(params['sigma']), size=N)
    trajs_grw_y[t + 1] = q_from_offset_log(z_next)
    z = np.log(trajs_grw_y[t + 1] + EPS)
registry_row, parameter_rows = save_model_output(model_tag, trajs_grw_y, MODEL_SEEDS[model_tag])
simulation_registry_rows.append(registry_row)
parameter_registry_rows.extend(parameter_rows)
print(MODEL_NAMES[model_tag], trajs_grw_y.shape)


GRW-Y (21, 50000)


In [7]:
# Cell 7: simulate single-estimation GRW

model_tag = 'grw_global'
rng = np.random.default_rng(MODEL_SEEDS[model_tag])
grw_global_params = read_one(model_tag, 'global-parameters.csv')
grw_global_initialization_params = read_one(model_tag, 'initialization-parameters.csv')
mu = row_value(grw_global_params, 'mu', 'mu_log_delta')
sigma = row_value(grw_global_params, 'sigma', 'sigma_log_delta')
scale = row_value(grw_global_initialization_params, 'scale', 'alpha_q0', 'q0_exponential_scale')
trajs_grw_global = np.empty((Y + 1, N), dtype=float)
trajs_grw_global[0] = rng.exponential(scale=scale, size=N)
z = np.log(trajs_grw_global[0] + EPS)
for t in range(Y):
    z = z + rng.normal(mu, sigma, size=N)
    trajs_grw_global[t + 1] = q_from_offset_log(z)
registry_row, parameter_rows = save_model_output(model_tag, trajs_grw_global, MODEL_SEEDS[model_tag])
simulation_registry_rows.append(registry_row)
parameter_registry_rows.extend(parameter_rows)
print(MODEL_NAMES[model_tag], trajs_grw_global.shape)


GRW-G (21, 50000)


In [8]:
# Cell 8: simulate AR(1)-GRW-Y

model_tag = 'ar1_y'
rng = np.random.default_rng(MODEL_SEEDS[model_tag])
ar1_y_params = read_table(model_tag, 'yearwise-parameters.csv').sort_values('transition_year')
ar1_y_global_params = read_table(model_tag, 'global-parameters.csv')
ar1_y_initialization_params = read_one(model_tag, 'initialization-parameters.csv')
if ar1_y_params['transition_year'].tolist() != list(range(Y)):
    raise AssertionError('AR(1)-GRW-Y parameters must cover transition years 0-19')
trajs_ar1_y = np.empty((Y + 1, N), dtype=float)
trajs_ar1_y[0] = rng.exponential(scale=float(ar1_y_initialization_params['scale']), size=N)
z = np.log(trajs_ar1_y[0] + EPS)
for _, params in ar1_y_params.iterrows():
    z_next = float(params['intercept']) + float(params['beta']) * z + rng.normal(0, float(params['sigma_resid']), size=N)
    q_next = q_from_offset_log(z_next)
    t = int(params['transition_year'])
    trajs_ar1_y[t + 1] = q_next
    z = np.log(q_next + EPS)
registry_row, parameter_rows = save_model_output(model_tag, trajs_ar1_y, MODEL_SEEDS[model_tag])
simulation_registry_rows.append(registry_row)
parameter_registry_rows.extend(parameter_rows)
print(MODEL_NAMES[model_tag], trajs_ar1_y.shape)


AR(1)-GRW-Y (21, 50000)


In [9]:
# Cell 9: simulate AR(1)-GRW-S

model_tag = 'ar1_s'
rng = np.random.default_rng(MODEL_SEEDS[model_tag])
ar1_s_params = read_table(model_tag, 'stagewise-parameters.csv')
ar1_s_global_params = read_table(model_tag, 'global-parameters.csv')
ar1_s_initialization_params = read_one(model_tag, 'initialization-parameters.csv')
ar1_s_by_year = stage_rows_by_year(ar1_s_params)
trajs_ar1_s = np.empty((Y + 1, N), dtype=float)
trajs_ar1_s[0] = rng.exponential(scale=float(ar1_s_initialization_params['scale']), size=N)
z = np.log(trajs_ar1_s[0] + EPS)
for t, params in enumerate(ar1_s_by_year):
    z_next = float(params['intercept']) + float(params['beta']) * z + rng.normal(0, float(params['sigma_resid']), size=N)
    trajs_ar1_s[t + 1] = q_from_offset_log(z_next)
    z = np.log(trajs_ar1_s[t + 1] + EPS)
registry_row, parameter_rows = save_model_output(model_tag, trajs_ar1_s, MODEL_SEEDS[model_tag])
simulation_registry_rows.append(registry_row)
parameter_registry_rows.extend(parameter_rows)
print(MODEL_NAMES[model_tag], trajs_ar1_s.shape)


AR(1)-GRW-S (21, 50000)


In [10]:
# Cell 10: simulate single-estimation GRW with stagewise AR(1)

model_tag = 'ar1_s_globalinit'
rng = np.random.default_rng(MODEL_SEEDS[model_tag])
ar1_s_globalinit_params = read_table(model_tag, 'stagewise-parameters.csv')
ar1_s_globalinit_global_params = read_table(model_tag, 'global-parameters.csv')
ar1_s_globalinit_initialization_params = read_one(model_tag, 'initialization-parameters.csv')
ar1_s_globalinit_by_year = stage_rows_by_year(ar1_s_globalinit_params)
scale = row_value(ar1_s_globalinit_initialization_params, 'scale', 'alpha_q0', 'q0_exponential_scale')
trajs_ar1_s_globalinit = np.empty((Y + 1, N), dtype=float)
trajs_ar1_s_globalinit[0] = rng.exponential(scale=scale, size=N)
z = np.log(trajs_ar1_s_globalinit[0] + EPS)
for t, params in enumerate(ar1_s_globalinit_by_year):
    z = float(params['intercept']) + float(params['beta']) * z + rng.normal(0, float(params['sigma_resid']), size=N)
    trajs_ar1_s_globalinit[t + 1] = q_from_offset_log(z)
    z = np.log(trajs_ar1_s_globalinit[t + 1] + EPS)
registry_row, parameter_rows = save_model_output(model_tag, trajs_ar1_s_globalinit, MODEL_SEEDS[model_tag])
simulation_registry_rows.append(registry_row)
parameter_registry_rows.extend(parameter_rows)
print(MODEL_NAMES[model_tag], trajs_ar1_s_globalinit.shape)


AR(1)-GRW-S-G (21, 50000)


In [11]:
# Cell 11: hurdle AR(1) simulation helper

def simulate_hurdle_ar1(model_tag, productivity_dependent_dropout, rng):
    hurdle_params = read_table(model_tag, 'stagewise-hurdle-parameters.csv')
    intensity_params = read_table(model_tag, 'stagewise-positive-ar1-parameters.csv')
    global_params = read_one(model_tag, 'global-positive-ar1-parameters.csv')
    initialization = read_one(model_tag, 'initialization-parameters.csv')
    hurdle_by_year = stage_rows_by_year(hurdle_params)
    intensity_lookup = intensity_params.set_index('stage').to_dict('index')
    dropout_lookup = None
    if productivity_dependent_dropout:
        dropout_params = read_table(model_tag, 'dropout-parameters.csv')
        dropout_lookup = dropout_params.set_index('stage').to_dict('index')
    trajectories = np.zeros((Y + 1, N), dtype=float)
    active = np.zeros((Y + 1, N), dtype=bool)
    active[0] = rng.random(N) < float(initialization['p_init_active'])
    trajectories[0, active[0]] = positive_exponential(float(initialization['positive_q0_exponential_scale']), int(active[0].sum()), rng)
    for t, hurdle in enumerate(hurdle_by_year):
        stage = str(hurdle['stage'])
        current_active = active[t]
        current_inactive = ~current_active
        next_active = np.zeros(N, dtype=bool)
        p_restart = np.clip(float(hurdle['p_01']), 0, 1)
        next_active[current_inactive] = rng.random(int(current_inactive.sum())) < p_restart
        if productivity_dependent_dropout:
            dropout = dropout_lookup[stage]
            eta = float(dropout['dropout_intercept']) - float(dropout['dropout_gamma_positive']) * trajectories[t, current_active] / float(dropout['productivity_scale'])
            p_dropout = logistic(eta)
            next_active[current_active] = rng.random(int(current_active.sum())) >= p_dropout
        else:
            p_stay = np.clip(float(hurdle['p_11']), 0, 1)
            next_active[current_active] = rng.random(int(current_active.sum())) < p_stay
        active[t + 1] = next_active
        restarted = current_inactive & next_active
        continued = current_active & next_active
        trajectories[t + 1, restarted] = positive_exponential(float(initialization['restart_exponential_scale']), int(restarted.sum()), rng)
        if continued.any():
            params = resolve_params(intensity_lookup, global_params, stage, ['intercept', 'beta', 'sigma_resid'])
            z_next = float(params['intercept']) + float(params['beta']) * np.log(trajectories[t, continued]) + rng.normal(0, float(params['sigma_resid']), size=int(continued.sum()))
            trajectories[t + 1, continued] = positive_from_log(z_next)
    return (trajectories, active)


In [12]:
# Cell 12: simulate Hurdle-AR(1)-GRW-S

model_tag = 'hurdle_ar1_s'
rng = np.random.default_rng(MODEL_SEEDS[model_tag])
trajs_hurdle_ar1_s, active_hurdle_ar1_s = simulate_hurdle_ar1(model_tag, productivity_dependent_dropout=False, rng=rng)
registry_row, parameter_rows = save_model_output(model_tag, trajs_hurdle_ar1_s, MODEL_SEEDS[model_tag], active=active_hurdle_ar1_s)
simulation_registry_rows.append(registry_row)
parameter_registry_rows.extend(parameter_rows)
print(MODEL_NAMES[model_tag], trajs_hurdle_ar1_s.shape)


Hurdle-AR(1)-GRW-S (21, 50000)


In [13]:
# Cell 13: simulate Hurdle-AR(1)-GRW-S-P

model_tag = 'hurdle_ar1_s_p'
rng = np.random.default_rng(MODEL_SEEDS[model_tag])
trajs_hurdle_ar1_s_p, active_hurdle_ar1_s_p = simulate_hurdle_ar1(model_tag, productivity_dependent_dropout=True, rng=rng)
registry_row, parameter_rows = save_model_output(model_tag, trajs_hurdle_ar1_s_p, MODEL_SEEDS[model_tag], active=active_hurdle_ar1_s_p)
simulation_registry_rows.append(registry_row)
parameter_registry_rows.extend(parameter_rows)
print(MODEL_NAMES[model_tag], trajs_hurdle_ar1_s_p.shape)


Hurdle-AR(1)-GRW-S-P (21, 50000)


In [14]:
# Cell 14: simulate Hurdle-AR(3)-GRW-S-P

model_tag = 'hurdle_ar3_s_p'
rng = np.random.default_rng(MODEL_SEEDS[model_tag])
hurdle_params = read_table(model_tag, 'stagewise-hurdle-parameters.csv')
stage_ar1_params = read_table(model_tag, 'stagewise-positive-ar1-warm-start-parameters.csv')
stage_ar2_params = read_table(model_tag, 'stagewise-positive-ar2-warm-start-parameters.csv')
stage_ar3_params = read_table(model_tag, 'stagewise-positive-ar3-parameters.csv')
global_ar1 = read_one(model_tag, 'global-positive-ar1-warm-start-parameters.csv')
global_ar2 = read_one(model_tag, 'global-positive-ar2-warm-start-parameters.csv')
global_ar3 = read_one(model_tag, 'global-positive-ar3-parameters.csv')
dropout_params = read_table(model_tag, 'dropout-parameters.csv')
initialization = read_one(model_tag, 'initialization-parameters.csv')
hurdle_by_year = stage_rows_by_year(hurdle_params)
stage_ar1_lookup = stage_ar1_params.set_index('stage').to_dict('index')
stage_ar2_lookup = stage_ar2_params.set_index('stage').to_dict('index')
stage_ar3_lookup = stage_ar3_params.set_index('stage').to_dict('index')
dropout_lookup = dropout_params.set_index('stage').to_dict('index')
trajs_hurdle_ar3_s_p = np.zeros((Y + 1, N), dtype=float)
active_hurdle_ar3_s_p = np.zeros((Y + 1, N), dtype=bool)
run_length_hurdle_ar3_s_p = np.zeros((Y + 1, N), dtype=np.int16)
active_hurdle_ar3_s_p[0] = rng.random(N) < float(initialization['p_init_active'])
trajs_hurdle_ar3_s_p[0, active_hurdle_ar3_s_p[0]] = positive_exponential(float(initialization['positive_q0_exponential_scale']), int(active_hurdle_ar3_s_p[0].sum()), rng)
run_length_hurdle_ar3_s_p[0, active_hurdle_ar3_s_p[0]] = 1
for t, hurdle in enumerate(hurdle_by_year):
    stage = str(hurdle['stage'])
    current_active = active_hurdle_ar3_s_p[t]
    current_inactive = ~current_active
    next_active = np.zeros(N, dtype=bool)
    p_restart = np.clip(float(hurdle['p_01']), 0, 1)
    next_active[current_inactive] = rng.random(int(current_inactive.sum())) < p_restart
    dropout = dropout_lookup[stage]
    eta = float(dropout['dropout_intercept']) - float(dropout['dropout_gamma_positive']) * trajs_hurdle_ar3_s_p[t, current_active] / float(dropout['productivity_scale'])
    p_dropout = logistic(eta)
    next_active[current_active] = rng.random(int(current_active.sum())) >= p_dropout
    active_hurdle_ar3_s_p[t + 1] = next_active
    restarted = current_inactive & next_active
    continued = current_active & next_active
    trajs_hurdle_ar3_s_p[t + 1, restarted] = positive_exponential(float(initialization['restart_exponential_scale']), int(restarted.sum()), rng)
    run_length_hurdle_ar3_s_p[t + 1, restarted] = 1
    continued_ar1 = continued & (run_length_hurdle_ar3_s_p[t] == 1)
    continued_ar2 = continued & (run_length_hurdle_ar3_s_p[t] == 2)
    continued_ar3 = continued & (run_length_hurdle_ar3_s_p[t] >= 3)
    if continued_ar1.any():
        params = resolve_params(stage_ar1_lookup, global_ar1, stage, ['intercept', 'beta_1', 'sigma_resid'])
        z_t = np.log(trajs_hurdle_ar3_s_p[t, continued_ar1])
        z_next = float(params['intercept']) + float(params['beta_1']) * z_t + rng.normal(0, float(params['sigma_resid']), size=int(continued_ar1.sum()))
        trajs_hurdle_ar3_s_p[t + 1, continued_ar1] = positive_from_log(z_next)
    if continued_ar2.any():
        params = resolve_params(stage_ar2_lookup, global_ar2, stage, ['intercept', 'beta_1', 'beta_2', 'sigma_resid'])
        z_t = np.log(trajs_hurdle_ar3_s_p[t, continued_ar2])
        z_lag_1 = np.log(trajs_hurdle_ar3_s_p[t - 1, continued_ar2])
        z_next = float(params['intercept']) + float(params['beta_1']) * z_t + float(params['beta_2']) * z_lag_1 + rng.normal(0, float(params['sigma_resid']), size=int(continued_ar2.sum()))
        trajs_hurdle_ar3_s_p[t + 1, continued_ar2] = positive_from_log(z_next)
    if continued_ar3.any():
        params = resolve_params(stage_ar3_lookup, global_ar3, stage, ['intercept', 'beta_1', 'beta_2', 'beta_3', 'sigma_resid'])
        z_t = np.log(trajs_hurdle_ar3_s_p[t, continued_ar3])
        z_lag_1 = np.log(trajs_hurdle_ar3_s_p[t - 1, continued_ar3])
        z_lag_2 = np.log(trajs_hurdle_ar3_s_p[t - 2, continued_ar3])
        z_next = float(params['intercept']) + float(params['beta_1']) * z_t + float(params['beta_2']) * z_lag_1 + float(params['beta_3']) * z_lag_2 + rng.normal(0, float(params['sigma_resid']), size=int(continued_ar3.sum()))
        trajs_hurdle_ar3_s_p[t + 1, continued_ar3] = positive_from_log(z_next)
    run_length_hurdle_ar3_s_p[t + 1, continued] = run_length_hurdle_ar3_s_p[t, continued] + 1
registry_row, parameter_rows = save_model_output(model_tag, trajs_hurdle_ar3_s_p, MODEL_SEEDS[model_tag], active=active_hurdle_ar3_s_p, active_run_length=run_length_hurdle_ar3_s_p)
simulation_registry_rows.append(registry_row)
parameter_registry_rows.extend(parameter_rows)
print(MODEL_NAMES[model_tag], trajs_hurdle_ar3_s_p.shape)


Hurdle-AR(3)-GRW-S-P (21, 50000)


In [15]:
# Cell 15: output registries and final audit

simulation_registry = pd.DataFrame(simulation_registry_rows)
parameter_registry = pd.DataFrame(parameter_registry_rows)
simulation_registry.to_csv(OUTPUT / 'simulation-registry.csv', index=False)
parameter_registry.to_csv(OUTPUT / 'parameter-registry.csv', index=False)
if simulation_registry['model_tag'].tolist() != MODEL_ORDER:
    raise AssertionError('Simulation registry does not match the required model order')
for model_tag in MODEL_ORDER:
    model_output = OUTPUT / model_tag
    required = [model_output / 'trajectories.npy', model_output / 'year-summary.csv', model_output / 'transition-summary.csv', model_output / 'trajectory-sample.csv', model_output / 'trajectory-long-sample.csv', model_output / 'simulation-metadata.csv']
    missing = [path.name for path in required if not path.is_file()]
    if missing:
        raise AssertionError(f'{model_tag}: missing outputs {missing}')
print(f'Saved {len(simulation_registry)} simulated models to {OUTPUT}')
display(simulation_registry)
display(parameter_registry)


Saved 10 simulated models to /Users/samlunemagid/Desktop/SPAARW2/simulate/output


,model,model_tag,n_trajectories,n_years,model_seed,trajectory_file,active_state_file,active_run_length_file,year_summary_file,transition_summary_file,trajectory_long_sample_file,minimum,maximum,zero_fraction_year_20
0,ARW4,arw4,50000,21,7893443217372262338,arw4/trajectories.npy,,,arw4/year-summary.csv,arw4/transition-summary.csv,arw4/trajectory-long-sample.csv,0.000042,6.359971e+01,0.00000
1,Unfitted-GRW,unfitted_grw,50000,21,12241077368650865958,unfitted_grw/trajectories.npy,,,unfitted_grw/year-summary.csv,unfitted_grw/transition-summary.csv,unfitted_grw/trajectory-long-sample.csv,0.000052,4.065087e+04,0.00000
2,GRW-Y,grw_y,50000,21,14512579941737846081,grw_y/trajectories.npy,,,grw_y/year-summary.csv,grw_y/transition-summary.csv,grw_y/trajectory-long-sample.csv,0.000000,6.397973e+09,0.10662
3,GRW-G,grw_global,50000,21,13939329440548590132,grw_global/trajectories.npy,,,grw_global/year-summary.csv,grw_global/transition-summary.csv,grw_global/trajectory-long-sample.csv,0.000000,6.154307e+08,0.30756
4,AR(1)-GRW-Y,ar1_y,50000,21,16860308996171422247,ar1_y/trajectories.npy,,,ar1_y/year-summary.csv,ar1_y/transition-summary.csv,ar1_y/trajectory-long-sample.csv,0.000000,1.074539e+03,0.03380
5,AR(1)-GRW-S,ar1_s,50000,21,10364501871564685049,ar1_s/trajectories.npy,,,ar1_s/year-summary.csv,ar1_s/transition-summary.csv,ar1_s/trajectory-long-sample.csv,0.000000,1.022662e+03,0.02466
6,AR(1)-GRW-S-G,ar1_s_globalinit,50000,21,13938879905710472210,ar1_s_globalinit/trajectories.npy,,,ar1_s_globalinit/year-summary.csv,ar1_s_globalinit/transition-summary.csv,ar1_s_globalinit/trajectory-long-sample.csv,0.000000,7.986845e+02,0.02484
7,Hurdle-AR(1)-GRW-S,hurdle_ar1_s,50000,21,1702118621661579778,hurdle_ar1_s/trajectories.npy,hurdle_ar1_s/active-states.npy,,hurdle_ar1_s/year-summary.csv,hurdle_ar1_s/transition-summary.csv,hurdle_ar1_s/trajectory-long-sample.csv,0.000000,2.813041e+02,0.13982
8,Hurdle-AR(1)-GRW-S-P,hurdle_ar1_s_p,50000,21,14245296302883193865,hurdle_ar1_s_p/trajectories.npy,hurdle_ar1_s_p/active-states.npy,,hurdle_ar1_s_p/year-summary.csv,hurdle_ar1_s_p/transition-summary.csv,hurdle_ar1_s_p/trajectory-long-sample.csv,0.000000,4.155771e+02,0.15486
9,Hurdle-AR(3)-GRW-S-P,hurdle_ar3_s_p,50000,21,18018853576801515496,hurdle_ar3_s_p/trajectories.npy,hurdle_ar3_s_p/active-states.npy,hurdle_ar3_s_p/active-run-length.npy,hurdle_ar3_s_p/year-summary.csv,hurdle_ar3_s_p/transition-summary.csv,hurdle_ar3_s_p/trajectory-long-sample.csv,0.000000,3.294540e+02,0.15964


,model_tag,file,relative_path,rows,columns
0,arw4,global-parameters.csv,arw4/parameters/global-parameters.csv,1,"stage,n,alpha,mode_beta,nll,mean_q_now,mean_q_..."
1,arw4,initialization-parameters.csv,arw4/parameters/initialization-parameters.csv,1,"distribution,n,scale"
2,arw4,stage-parameters.csv,arw4/parameters/stage-parameters.csv,4,"stage,transition_year_start,transition_year_en..."
3,unfitted_grw,fixed-parameters.csv,unfitted_grw/parameters/fixed-parameters.csv,4,"parameter,value"
4,grw_y,global-parameters.csv,grw_y/parameters/global-parameters.csv,1,"model,n,mu,sigma,var"
5,grw_y,initialization-parameters.csv,grw_y/parameters/initialization-parameters.csv,1,"distribution,n,scale"
6,grw_y,yearwise-parameters.csv,grw_y/parameters/yearwise-parameters.csv,20,"transition_year,n,mu,sigma,var"
7,grw_global,global-parameters.csv,grw_global/parameters/global-parameters.csv,1,"model,n,mu,sigma,var"
8,grw_global,initialization-parameters.csv,grw_global/parameters/initialization-parameter...,1,"distribution,n,scale"
9,ar1_y,global-parameters.csv,ar1_y/parameters/global-parameters.csv,1,"model,n,intercept,beta,sigma_resid,var_resid,i..."
